# Results for model: meta-llama/Llama-3.1-8B-Instruct

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np

# Load data
df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

# Define rolling window size
window_size = 100

# Define TOP_QUANTILE
TOP_QUANTILE = 0.15

# Feature Engineering
df = df.with_columns([
    pl.col('date_id').rolling(window_size).apply(
        lambda x: pl.col('feature_00').over(x).quantile(TOP_QUANTILE)
    ).alias('feature_00_top_quantile')
])

# Calculate feature_00 relative to responder_6
df = df.with_columns([
    (pl.col('feature_00') - pl.col('responder_6')).alias('feature_00_relative')
])

# Split data into training and validation sets
train_df = df.filter(pl.col('date_id') < 0.8 * df['date_id'].max())
val_df = df.filter(pl.col('date_id') >= 0.8 * df['date_id'].max())

# Define XGBoost Regressor
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', max_depth=5, learning_rate=0.1, n_estimators=100)

# Train XGBoost Regressor
xgb_model.fit(train_df.select(['feature_00_relative', 'responder_6']).to_pandas().values, train_df['responder_6'].to_pandas().values)

# Make predictions on validation set
y_pred = xgb_model.predict(train_df.select(['feature_00_relative']).to_pandas().values)

# Evaluate model
print('Training RMSE:', np.sqrt(np.mean((y_pred - train_df['responder_6'].to_pandas().values) ** 2)))
print('Validation RMSE:', np.sqrt(np.mean((xgb_model.predict(val_df.select(['feature_00_relative']).to_pandas().values) - val_df['responder_6'].to_pandas().values) ** 2)))